# 01c Prepare RegGS Scenes (405841, 3 Segments)

Goal:

1. adapt `405841` into three RegGS-ready scenes (A/B/C);
2. split by motion stages with boundaries at frame 25 and 70;
3. preserve per-segment geometric continuity;
4. export clean `intrinsics.json` and `cameras.json` for each segment;
5. generate per-segment sparse `train/test` split (1/10) and verify layout.

Canonical prepared-data path:

```text
dataset/405841/part2/<export_scene_name>/scene_A
dataset/405841/part2/<export_scene_name>/scene_B
dataset/405841/part2/<export_scene_name>/scene_C
```

Segment definition (half-open intervals):
- scene A: [0, 25)
- scene B: [25, 70)
- scene C: [70, N)

In [1]:
from pathlib import Path
import csv
import json
import math
import os
import shutil
import struct
import numpy as np
from PIL import Image

CV_ROOT = Path('~/CV_Project').expanduser()
DATASET_ROOT = CV_ROOT / 'dataset'

SOURCE_ROOT = DATASET_ROOT / '405841'
SOURCE_FRONT_ROOT = SOURCE_ROOT / 'FRONT'
SOURCE_IMAGE_DIR = SOURCE_FRONT_ROOT / 'rgb'
SOURCE_CALIB_DIR = SOURCE_FRONT_ROOT / 'calib'
SOURCE_GT_DIR = SOURCE_FRONT_ROOT / 'gt'
SOURCE_GT_OLD_DIR = SOURCE_FRONT_ROOT / 'gt_old'
PART2_ROOT = SOURCE_ROOT / 'part2'

PART2_ROOT.mkdir(parents=True, exist_ok=True)

print('SOURCE_ROOT =', SOURCE_ROOT)
print('SOURCE_IMAGE_DIR =', SOURCE_IMAGE_DIR)
print('SOURCE_CALIB_DIR =', SOURCE_CALIB_DIR)
print('SOURCE_GT_OLD_DIR =', SOURCE_GT_OLD_DIR)
print('PART2_ROOT =', PART2_ROOT)

SOURCE_ROOT = /home/bzhang512/CV_Project/dataset/405841
SOURCE_IMAGE_DIR = /home/bzhang512/CV_Project/dataset/405841/FRONT/rgb
SOURCE_CALIB_DIR = /home/bzhang512/CV_Project/dataset/405841/FRONT/calib
SOURCE_GT_OLD_DIR = /home/bzhang512/CV_Project/dataset/405841/FRONT/gt_old
PART2_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2


## 1. Export configuration

In [2]:
EXPORT_SCENE_NAME = 'reggs_405841_3seg_25_70'
EXPORT_ROOT = PART2_ROOT / EXPORT_SCENE_NAME

# Segment boundaries are half-open: [start, end). Use None as end for the final segment.
SEGMENT_SPECS = [
    ('A', 0, 25),
    ('B', 25, 70),
    ('C', 70, None),
]

# Sparse-view selection inside each segment.
SPARSE_STRIDE = 10
MIN_TRAIN_VIEWS = 3
INCLUDE_LAST_FRAME_IN_TRAIN = True

# Use gt_old as default because it contains full 4x4 pose with translation.
POSE_SOURCE = 'gt_old'  # choices: 'gt_old', 'gt'

# Set None to keep original size (symlink). Set (W, H) to export resized PNGs.
TARGET_SIZE = (960, 640)
OVERWRITE_EXPORT = True

print('EXPORT_ROOT =', EXPORT_ROOT)
print('SEGMENT_SPECS =', SEGMENT_SPECS)
print('SPARSE_STRIDE =', SPARSE_STRIDE)
print('MIN_TRAIN_VIEWS =', MIN_TRAIN_VIEWS)
print('INCLUDE_LAST_FRAME_IN_TRAIN =', INCLUDE_LAST_FRAME_IN_TRAIN)
print('POSE_SOURCE =', POSE_SOURCE)
print('TARGET_SIZE =', TARGET_SIZE)
print('OVERWRITE_EXPORT =', OVERWRITE_EXPORT)

EXPORT_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70
SEGMENT_SPECS = [('A', 0, 25), ('B', 25, 70), ('C', 70, None)]
SPARSE_STRIDE = 10
MIN_TRAIN_VIEWS = 3
INCLUDE_LAST_FRAME_IN_TRAIN = True
POSE_SOURCE = gt_old
TARGET_SIZE = (960, 640)
OVERWRITE_EXPORT = True


## 2. Helpers

In [3]:
def read_png_size(path: Path):
    png_sig = bytes.fromhex('89504E470D0A1A0A')
    with path.open('rb') as f:
        sig = f.read(8)
        if sig != png_sig:
            raise ValueError(f'Not a PNG file: {path}')
        _length = struct.unpack('>I', f.read(4))[0]
        chunk_type = f.read(4)
        if chunk_type != b'IHDR':
            raise ValueError(f'Invalid PNG header: {path}')
        width, height = struct.unpack('>II', f.read(8))
    return width, height


def safe_reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def symlink_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)


def parse_calib_intrinsics(calib_path: Path):
    text = calib_path.read_text(encoding='utf-8').strip().splitlines()
    if len(text) < 2:
        raise ValueError(f'Invalid calib file (too short): {calib_path}')

    vals = {}
    for token in text[1].replace(':', ' ').split():
        if token in {'fx', 'fy', 'cx', 'cy'}:
            vals[token] = None

    parts = text[1].replace(':', '').split()
    # expected pattern: fx <v> fy <v> cx <v> cy <v>
    for i in range(0, len(parts) - 1, 2):
        k = parts[i]
        v = parts[i + 1]
        if k in {'fx', 'fy', 'cx', 'cy'}:
            vals[k] = float(v)

    if any(vals[k] is None for k in ('fx', 'fy', 'cx', 'cy')):
        raise ValueError(f'Cannot parse intrinsics from: {calib_path}')
    return vals['fx'], vals['fy'], vals['cx'], vals['cy']


def read_pose_4x4(path: Path):
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines() if ln.strip()]
    if len(lines) < 4:
        raise ValueError(f'Pose file must have 4 rows: {path}')
    rows = []
    for i in range(4):
        vals = [float(x) for x in lines[i].split()]
        if len(vals) != 4:
            raise ValueError(f'Pose row {i} in {path} does not have 4 columns')
        rows.append(vals)
    mat = np.array(rows, dtype=float)
    return mat


def rotation_to_quat_xyzw(R: np.ndarray):
    m00, m01, m02 = R[0, 0], R[0, 1], R[0, 2]
    m10, m11, m12 = R[1, 0], R[1, 1], R[1, 2]
    m20, m21, m22 = R[2, 0], R[2, 1], R[2, 2]
    tr = m00 + m11 + m22

    if tr > 0.0:
        S = math.sqrt(tr + 1.0) * 2.0
        qw = 0.25 * S
        qx = (m21 - m12) / S
        qy = (m02 - m20) / S
        qz = (m10 - m01) / S
    elif (m00 > m11) and (m00 > m22):
        S = math.sqrt(1.0 + m00 - m11 - m22) * 2.0
        qw = (m21 - m12) / S
        qx = 0.25 * S
        qy = (m01 + m10) / S
        qz = (m02 + m20) / S
    elif m11 > m22:
        S = math.sqrt(1.0 + m11 - m00 - m22) * 2.0
        qw = (m02 - m20) / S
        qx = (m01 + m10) / S
        qy = 0.25 * S
        qz = (m12 + m21) / S
    else:
        S = math.sqrt(1.0 + m22 - m00 - m11) * 2.0
        qw = (m10 - m01) / S
        qx = (m02 + m20) / S
        qy = (m12 + m21) / S
        qz = 0.25 * S

    q = np.array([qx, qy, qz, qw], dtype=float)
    q = q / np.linalg.norm(q)
    return q.tolist()


def choose_train_test_local_ids(n_frames: int, stride: int, min_train: int, include_last: bool):
    if n_frames <= 0:
        return [], []

    train_ids = list(range(0, n_frames, stride))
    if include_last:
        train_ids.append(n_frames - 1)
    train_ids = sorted(set(train_ids))

    if len(train_ids) < min_train:
        extra = np.linspace(0, n_frames - 1, min_train).astype(int).tolist()
        train_ids = sorted(set(train_ids + extra))

    test_ids = [i for i in range(n_frames) if i not in set(train_ids)]
    return train_ids, test_ids


def write_json(path: Path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')

## 3. Load source scene (405841)

In [4]:
def load_405841_source():
    image_paths = sorted(SOURCE_IMAGE_DIR.glob('*.png'))
    calib_paths = sorted(SOURCE_CALIB_DIR.glob('*.txt'))

    if POSE_SOURCE == 'gt_old':
        pose_dir = SOURCE_GT_OLD_DIR
    elif POSE_SOURCE == 'gt':
        pose_dir = SOURCE_GT_DIR
    else:
        raise ValueError(f'Unsupported POSE_SOURCE: {POSE_SOURCE}')

    pose_paths = sorted(pose_dir.glob('*.txt'))

    if not image_paths:
        raise FileNotFoundError(f'No images found in {SOURCE_IMAGE_DIR}')
    if len(image_paths) != len(calib_paths) or len(image_paths) != len(pose_paths):
        raise ValueError(
            f'Count mismatch: images={len(image_paths)}, calib={len(calib_paths)}, poses={len(pose_paths)}'
        )

    width, height = read_png_size(image_paths[0])
    fx, fy, cx, cy = parse_calib_intrinsics(calib_paths[0])
    intrinsics = {
        'fx': fx / width,
        'fy': fy / height,
        'cx': cx / width,
        'cy': cy / height,
    }

    # Verify intrinsics consistency across all frames.
    for p in calib_paths[1:]:
        _fx, _fy, _cx, _cy = parse_calib_intrinsics(p)
        if not (
            abs(_fx - fx) < 1e-9
            and abs(_fy - fy) < 1e-9
            and abs(_cx - cx) < 1e-9
            and abs(_cy - cy) < 1e-9
        ):
            raise ValueError(f'Per-frame intrinsics are not constant: {p.name}')

    records = []
    for i, (img, cal, pose) in enumerate(zip(image_paths, calib_paths, pose_paths)):
        if img.stem != cal.stem or img.stem != pose.stem:
            raise ValueError(f'Stem mismatch at index {i}: {img.name}, {cal.name}, {pose.name}')
        records.append({
            'global_idx': i,
            'stem': img.stem,
            'image_path': img,
            'calib_path': cal,
            'pose_path': pose,
        })

    return {
        'records': records,
        'source_width': width,
        'source_height': height,
        'intrinsics': intrinsics,
        'pose_dir': pose_dir,
    }


source = load_405841_source()
print('num frames =', len(source['records']))
print('source size =', (source['source_width'], source['source_height']))
print('normalized intrinsics =', source['intrinsics'])
print('pose dir =', source['pose_dir'])

num frames = 199
source size = (1920, 1280)
normalized intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
pose dir = /home/bzhang512/CV_Project/dataset/405841/FRONT/gt_old


## 4. Build segment indices

In [5]:
def build_segments(records, segment_specs):
    n = len(records)
    segments = []
    for seg_name, start, end in segment_specs:
        if end is None:
            end = n
        if not (0 <= start < end <= n):
            raise ValueError(f'Invalid segment range for {seg_name}: [{start}, {end}) with n={n}')
        seg_records = records[start:end]
        segments.append({
            'name': seg_name,
            'start': start,
            'end': end,
            'records': seg_records,
        })
    return segments


segments = build_segments(source['records'], SEGMENT_SPECS)

for seg in segments:
    seg_records = seg['records']
    print(f"scene {seg['name']}: [{seg['start']}, {seg['end']}) | num_frames={len(seg_records)}")
    print('  first global frame =', seg_records[0]['global_idx'], seg_records[0]['image_path'].name)
    print('  last  global frame =', seg_records[-1]['global_idx'], seg_records[-1]['image_path'].name)

scene A: [0, 25) | num_frames=25
  first global frame = 0 000000.png
  last  global frame = 24 000024.png
scene B: [25, 70) | num_frames=45
  first global frame = 25 000025.png
  last  global frame = 69 000069.png
scene C: [70, 199) | num_frames=129
  first global frame = 70 000070.png
  last  global frame = 198 000198.png


## 5. Export each segment to RegGS scene format

In [6]:
SOURCE_SIZE = (source['source_width'], source['source_height'])
if TARGET_SIZE is None:
    EXPORT_WIDTH, EXPORT_HEIGHT = SOURCE_SIZE
else:
    if len(TARGET_SIZE) != 2:
        raise ValueError(f'TARGET_SIZE must be (W, H), got: {TARGET_SIZE}')
    EXPORT_WIDTH = int(TARGET_SIZE[0])
    EXPORT_HEIGHT = int(TARGET_SIZE[1])
    if EXPORT_WIDTH <= 0 or EXPORT_HEIGHT <= 0:
        raise ValueError(f'Invalid TARGET_SIZE: {TARGET_SIZE}')

USE_SYMLINKS = (EXPORT_WIDTH, EXPORT_HEIGHT) == SOURCE_SIZE


def write_intrinsics_json(export_root: Path, intrinsics: dict):
    payload = {
        'fx': float(intrinsics['fx']),
        'fy': float(intrinsics['fy']),
        'cx': float(intrinsics['cx']),
        'cy': float(intrinsics['cy']),
    }
    write_json(export_root / 'intrinsics.json', payload)


def export_image(src: Path, dst: Path, use_symlinks: bool, export_width: int, export_height: int):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if use_symlinks:
        symlink_file(src, dst)
        return

    with Image.open(src) as im:
        resized = im.convert('RGB').resize((export_width, export_height), Image.LANCZOS)
        resized.save(dst, format='PNG')


def make_export_camera(local_idx: int, global_idx: int, pose_w2c: np.ndarray, intrinsics: dict):
    R = pose_w2c[:3, :3]
    t = pose_w2c[:3, 3]
    quat_xyzw = rotation_to_quat_xyzw(R)
    return {
        'cam_id': int(local_idx),
        'cam_quat': [float(x) for x in quat_xyzw],
        'cam_trans': [float(x) for x in t.tolist()],
        'cx': float(intrinsics['cx']),
        'cy': float(intrinsics['cy']),
        'fx': float(intrinsics['fx']),
        'fy': float(intrinsics['fy']),
        'image_name': f'{local_idx:04d}.png',
        'timestamp': int(global_idx),
    }


def export_segment_scene(segment: dict, source_dict: dict, root: Path):
    seg_name = segment['name']
    seg_records = segment['records']
    seg_root = root / f'scene_{seg_name}'

    if seg_root.exists() and not OVERWRITE_EXPORT:
        raise FileExistsError(f'{seg_root} already exists and overwrite=False')
    safe_reset_dir(seg_root)

    image_dir = seg_root / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)

    export_cameras = []
    index_map = []

    for local_idx, rec in enumerate(seg_records):
        src = rec['image_path']
        dst = image_dir / f'{local_idx:04d}.png'
        export_image(src, dst, USE_SYMLINKS, EXPORT_WIDTH, EXPORT_HEIGHT)

        pose = read_pose_4x4(rec['pose_path'])
        export_cameras.append(
            make_export_camera(local_idx, rec['global_idx'], pose, source_dict['intrinsics'])
        )

        index_map.append({
            'local_idx': int(local_idx),
            'global_idx': int(rec['global_idx']),
            'local_image_name': f'{local_idx:04d}.png',
            'source_image_name': rec['image_path'].name,
            'source_pose_file': rec['pose_path'].name,
            'source_calib_file': rec['calib_path'].name,
        })

    write_intrinsics_json(seg_root, source_dict['intrinsics'])
    write_json(seg_root / 'cameras.json', export_cameras)
    write_json(seg_root / 'index_map.json', index_map)

    train_ids_local, test_ids_local = choose_train_test_local_ids(
        n_frames=len(seg_records),
        stride=SPARSE_STRIDE,
        min_train=MIN_TRAIN_VIEWS,
        include_last=INCLUDE_LAST_FRAME_IN_TRAIN,
    )

    train_ids_global = [seg_records[i]['global_idx'] for i in train_ids_local]
    test_ids_global = [seg_records[i]['global_idx'] for i in test_ids_local]

    split_payload = {
        'scene_name': f'scene_{seg_name}',
        'segment_range': {
            'start': int(segment['start']),
            'end_exclusive': int(segment['end']),
        },
        'num_frames': int(len(seg_records)),
        'sparse_stride': int(SPARSE_STRIDE),
        'min_train_views': int(MIN_TRAIN_VIEWS),
        'include_last_frame_in_train': bool(INCLUDE_LAST_FRAME_IN_TRAIN),
        'train_ids_local': [int(x) for x in train_ids_local],
        'test_ids_local': [int(x) for x in test_ids_local],
        'train_ids_global': [int(x) for x in train_ids_global],
        'test_ids_global': [int(x) for x in test_ids_global],
    }

    write_json(seg_root / 'split_train_test.json', split_payload)

    return {
        'scene_name': f'scene_{seg_name}',
        'segment_name': seg_name,
        'global_start': int(segment['start']),
        'global_end_exclusive': int(segment['end']),
        'num_frames': int(len(seg_records)),
        'num_train': int(len(train_ids_local)),
        'num_test': int(len(test_ids_local)),
        'train_ratio': float(len(train_ids_local) / len(seg_records)),
        'export_root': str(seg_root),
        'train_ids_global_head': train_ids_global[:8],
        'export_width': int(EXPORT_WIDTH),
        'export_height': int(EXPORT_HEIGHT),
        'use_symlinks': bool(USE_SYMLINKS),
    }


if EXPORT_ROOT.exists() and OVERWRITE_EXPORT:
    safe_reset_dir(EXPORT_ROOT)
else:
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

scene_summaries = [export_segment_scene(seg, source, EXPORT_ROOT) for seg in segments]
print('SOURCE_SIZE =', SOURCE_SIZE)
print('EXPORT_SIZE =', (EXPORT_WIDTH, EXPORT_HEIGHT))
print('USE_SYMLINKS =', USE_SYMLINKS)

for row in scene_summaries:
    print(json.dumps(row, indent=2))

SOURCE_SIZE = (1920, 1280)
EXPORT_SIZE = (960, 640)
USE_SYMLINKS = False
{
  "scene_name": "scene_A",
  "segment_name": "A",
  "global_start": 0,
  "global_end_exclusive": 25,
  "num_frames": 25,
  "num_train": 4,
  "num_test": 21,
  "train_ratio": 0.16,
  "export_root": "/home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/scene_A",
  "train_ids_global_head": [
    0,
    10,
    20,
    24
  ],
  "export_width": 960,
  "export_height": 640,
  "use_symlinks": false
}
{
  "scene_name": "scene_B",
  "segment_name": "B",
  "global_start": 25,
  "global_end_exclusive": 70,
  "num_frames": 45,
  "num_train": 6,
  "num_test": 39,
  "train_ratio": 0.13333333333333333,
  "export_root": "/home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/scene_B",
  "train_ids_global_head": [
    25,
    35,
    45,
    55,
    65,
    69
  ],
  "export_width": 960,
  "export_height": 640,
  "use_symlinks": false
}
{
  "scene_name": "scene_C",
  "segment_name": "C",
  "g

## 6. Verify per-segment exports

In [7]:
for row in scene_summaries:
    seg_root = Path(row['export_root'])
    images = sorted((seg_root / 'images').iterdir())
    cameras = json.loads((seg_root / 'cameras.json').read_text(encoding='utf-8'))
    intrinsics = json.loads((seg_root / 'intrinsics.json').read_text(encoding='utf-8'))
    split_payload = json.loads((seg_root / 'split_train_test.json').read_text(encoding='utf-8'))

    assert len(images) == len(cameras) == row['num_frames']
    assert images[0].name == '0000.png'
    assert cameras[0]['image_name'] == '0000.png'
    assert cameras[0]['cam_id'] == 0

    w0, h0 = read_png_size(images[0])
    assert w0 == row['export_width'] and h0 == row['export_height'], (
        f'Unexpected export size for {row["scene_name"]}: {(w0, h0)} vs {(row["export_width"], row["export_height"])}'
    )

    train_ids = split_payload['train_ids_local']
    test_ids = split_payload['test_ids_local']
    assert len(set(train_ids).intersection(set(test_ids))) == 0
    assert sorted(train_ids + test_ids) == list(range(row['num_frames']))

    print(f"{row['scene_name']} ok: frames={row['num_frames']} train={row['num_train']} test={row['num_test']}")
    print('  export_size =', (row['export_width'], row['export_height']))
    print('  use_symlinks =', row['use_symlinks'])
    print('  intrinsics =', intrinsics)
    print('  train_global_head =', split_payload['train_ids_global'][:8])

scene_A ok: frames=25 train=4 test=21
  export_size = (960, 640)
  use_symlinks = False
  intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
  train_global_head = [0, 10, 20, 24]
scene_B ok: frames=45 train=6 test=39
  export_size = (960, 640)
  use_symlinks = False
  intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
  train_global_head = [25, 35, 45, 55, 65, 69]
scene_C ok: frames=129 train=14 test=115
  export_size = (960, 640)
  use_symlinks = False
  intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
  train_global_head = [70, 80, 90, 100, 110, 120, 130, 140]


## 7. Global coverage and overlap checks

In [8]:
all_global_indices = []
for row in scene_summaries:
    seg_root = Path(row['export_root'])
    idx_map = json.loads((seg_root / 'index_map.json').read_text(encoding='utf-8'))
    globals_this_scene = [item['global_idx'] for item in idx_map]
    all_global_indices.extend(globals_this_scene)

expected = list(range(len(source['records'])))
assert sorted(all_global_indices) == expected, 'Global coverage mismatch across A/B/C segments.'
assert len(set(all_global_indices)) == len(all_global_indices), 'Found overlap among segment exports.'
print('Global coverage check passed for all segments.')

Global coverage check passed for all segments.


## 8. Write summary and manifest

In [9]:
summary_json_path = EXPORT_ROOT / 'scene_summary.json'
summary_csv_path = EXPORT_ROOT / 'scene_summary.csv'
manifest_path = EXPORT_ROOT / 'segment_manifest.json'

write_json(summary_json_path, scene_summaries)

with summary_csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'scene_name',
        'segment_name',
        'global_start',
        'global_end_exclusive',
        'num_frames',
        'num_train',
        'num_test',
        'train_ratio',
        'export_root',
        'export_width',
        'export_height',
        'use_symlinks',
    ])
    writer.writeheader()
    for row in scene_summaries:
        writer.writerow({k: row[k] for k in writer.fieldnames})

manifest = {
    'source_scene': '405841',
    'export_scene_name': EXPORT_SCENE_NAME,
    'pose_source': POSE_SOURCE,
    'source_size': {'width': int(SOURCE_SIZE[0]), 'height': int(SOURCE_SIZE[1])},
    'export_size': {'width': int(EXPORT_WIDTH), 'height': int(EXPORT_HEIGHT)},
    'resize_applied': bool(not USE_SYMLINKS),
    'intrinsics_convention': 'normalized_xy_by_export_size',
    'segment_specs': [
        {'name': n, 'start': s, 'end_exclusive': e if e is not None else len(source['records'])}
        for n, s, e in SEGMENT_SPECS
    ],
    'sparse_policy': {
        'stride': SPARSE_STRIDE,
        'min_train_views': MIN_TRAIN_VIEWS,
        'include_last_frame_in_train': INCLUDE_LAST_FRAME_IN_TRAIN,
    },
    'scenes': [
        {
            'scene_name': row['scene_name'],
            'export_root': row['export_root'],
            'split_file': str(Path(row['export_root']) / 'split_train_test.json'),
            'cameras_file': str(Path(row['export_root']) / 'cameras.json'),
            'intrinsics_file': str(Path(row['export_root']) / 'intrinsics.json'),
            'export_width': row['export_width'],
            'export_height': row['export_height'],
            'use_symlinks': row['use_symlinks'],
        }
        for row in scene_summaries
    ],
}
write_json(manifest_path, manifest)

print('wrote:', summary_json_path)
print('wrote:', summary_csv_path)
print('wrote:', manifest_path)
print('Done: 405841 segmented RegGS scenes are prepared.')

wrote: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/scene_summary.json
wrote: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/scene_summary.csv
wrote: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/segment_manifest.json
Done: 405841 segmented RegGS scenes are prepared.
